# FlowSure Data Pipeline — AI-Powered Ticketsysteem

**Vak:** ML Engineering & Ops  
**Inlevering:** Week 6 — Datapipeline onderdeel  
**Team:** [Vul jullie namen in]  

Dit notebook bouwt een end-to-end datapipeline voor het FlowSure klantenondersteuningsproject.
We verwerken twee datasets (Bitext en Twitter) via een medallion-architectuur (Bronze → Silver → Gold)
en demonstreren zowel batch- als streamingverwerking met PySpark op Databricks.

---

# 1. Setup

We importeren de PySpark-functies die we nodig hebben voor datavalidatie en -manipulatie.

In [0]:
from pyspark.sql.functions import col, count, when, isnan

# 2. Exploratory Data Analysis (EDA)

## 2.1 Bitext Dataset

We laden de Bitext Customer Support dataset rechtstreeks vanuit de Databricks Volume.
Deze dataset bevat gelabelde klantvragen met intent, categorie en voorbeeldantwoorden —
bedoeld als trainingsdata voor het **edge-model** (classificatie).

In [0]:
# Laad Bitext dataset en bekijk de structuur
df_bitext_raw = spark.read.csv(
    "/Volumes/flowsure/default/raw_files/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv",
    header=True,
    inferSchema=True,
)

print(f"Bitext: {df_bitext_raw.count():,} rijen, {len(df_bitext_raw.columns)} kolommen")
df_bitext_raw.printSchema()

Bitext: 71,963 rijen, 5 kolommen
root
 |-- flags: string (nullable = true)
 |-- instruction: string (nullable = true)
 |-- category: string (nullable = true)
 |-- intent: string (nullable = true)
 |-- response: string (nullable = true)



In [0]:
# Bekijk de eerste 5 rijen van Bitext
df_bitext_raw.show(5, truncate=80)

+--------------------------------------------------------------------------------+----------------------------------------------------------+--------+------------+--------------------------------------------------------------------------------+
|                                                                           flags|                                               instruction|category|      intent|                                                                        response|
+--------------------------------------------------------------------------------+----------------------------------------------------------+--------+------------+--------------------------------------------------------------------------------+
|                                                                               B|          question about cancelling order {{Order Number}}|   ORDER|cancel_order|I've understood you have a question regarding canceling order {{Order Number}...|
|                   

**Observatie:** De dataset bevat 71.963 rijen en 5 kolommen (`flags`, `instruction`, `category`, `intent`, `response`).
In de preview valt op dat rij 4 en 5 NULL-waarden hebben in `category`, `intent` en `response`.
De `flags`-kolom van die rijen bevat tekst die eigenlijk deel uitmaakt van het antwoord uit de vorige rij —
dit zijn kapotte rijen veroorzaakt door multiline-tekst in de CSV.

## 2.1b Twitter Dataset — Inladen

We laden de Twitter Customer Support dataset (~3M tweets).
Deze bevat ruwe conversaties tussen klanten en bedrijven — zonder labels.
Bedoeld als trainingsdata voor het **cloud-model** (antwoord genereren).

In [0]:
# Laad Twitter dataset en bekijk de structuur
df_twitter_raw = spark.read.csv(
    "/Volumes/flowsure/default/raw_files/archive/twcs/twcs.csv",
    header=True,
    inferSchema=True,
)

print(f"Twitter: {df_twitter_raw.count():,} rijen, {len(df_twitter_raw.columns)} kolommen")
df_twitter_raw.printSchema()

Twitter: 2,966,469 rijen, 7 kolommen
root
 |-- tweet_id: string (nullable = true)
 |-- author_id: string (nullable = true)
 |-- inbound: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- text: string (nullable = true)
 |-- response_tweet_id: string (nullable = true)
 |-- in_response_to_tweet_id: string (nullable = true)



In [0]:
# Bekijk de eerste 5 rijen van Twitter
df_twitter_raw.show(5, truncate=80)

+--------+----------+-------+------------------------------+--------------------------------------------------------------------------------+-----------------+-----------------------+
|tweet_id| author_id|inbound|                    created_at|                                                                            text|response_tweet_id|in_response_to_tweet_id|
+--------+----------+-------+------------------------------+--------------------------------------------------------------------------------+-----------------+-----------------------+
|       1|sprintcare|  False|Tue Oct 31 22:10:47 +0000 2017|@115712 I understand. I would like to assist you. We would need to get you in...|                2|                      3|
|       2|    115712|   True|Tue Oct 31 22:11:45 +0000 2017|                                   @sprintcare and how do you propose we do that|             NULL|                      1|
|       3|    115712|   True|Tue Oct 31 22:08:27 +0000 2017|@sprintcare I have s

**Observatie:** 2.966.469 rijen met 7 kolommen. Belangrijke velden: `text` (de tweet),
`inbound` (True = klant, False = support), en `response_tweet_id` / `in_response_to_tweet_id`
waarmee we conversatieparen kunnen reconstrueren. Let op: `created_at` is een string, geen timestamp —
die moeten we later converteren.

### Bitext — Datakwaliteit: hoeveel rijen zijn bruikbaar?

We tellen hoeveel rijen NULL-waarden hebben in de essentiële kolommen.
Rijen zonder `category`, `intent` of `instruction` zijn onbruikbaar voor classificatie.

In [0]:
# Hoeveel rijen zijn compleet vs kapot (NULLs in category of intent)?


total = df_bitext_raw.count()
nulls_category = df_bitext_raw.filter(col("category").isNull()).count()
nulls_intent = df_bitext_raw.filter(col("intent").isNull()).count()
nulls_instruction = df_bitext_raw.filter(col("instruction").isNull()).count()
clean_rows = df_bitext_raw.filter(col("category").isNotNull() & col("intent").isNotNull() & col("instruction").isNotNull()).count()

print(f"Totaal rijen:           {total:,}")
print(f"NULL in category:       {nulls_category:,}  ({nulls_category/total*100:.1f}%)")
print(f"NULL in intent:         {nulls_intent:,}  ({nulls_intent/total*100:.1f}%)")
print(f"NULL in instruction:    {nulls_instruction:,}  ({nulls_instruction/total*100:.1f}%)")
print(f"Volledig bruikbare rijen: {clean_rows:,}  ({clean_rows/total*100:.1f}%)")

Totaal rijen:           71,963
NULL in category:       37,956  (52.7%)
NULL in intent:         42,544  (59.1%)
NULL in instruction:    21,432  (29.8%)
Volledig bruikbare rijen: 29,419  (40.9%)


**Observatie:** Meer dan de helft van de rijen (52.7% in category, 59.1% in intent) bevat NULLs.
Dit klinkt ernstig, maar deze rijen zijn geen echte tickets — het zijn stukken van lange antwoorden
die over meerdere CSV-regels liepen. De **29.419 volledig bruikbare rijen** zijn ruim voldoende voor modeltraining.

### Bitext — Unieke categorieën en intents

We bekijken hoeveel unieke waarden er zijn in de bruikbare rijen.
Dit helpt ons te begrijpen hoe het classificatieprobleem eruitziet.

In [0]:
# Alleen de bruikbare rijen — hoeveel unieke categorieën en intents?
df_bitext_clean = df_bitext_raw.filter(
    col("category").isNotNull() & col("intent").isNotNull() & col("instruction").isNotNull()
)

n_categories = df_bitext_clean.select("category").distinct().count()
n_intents = df_bitext_clean.select("intent").distinct().count()

print(f"Bruikbare rijen: {df_bitext_clean.count():,}")
print(f"Unieke categorieën: {n_categories}")
print(f"Unieke intents: {n_intents}")
print(f"\n--- Categorieën ---")
df_bitext_clean.groupBy("category").count().orderBy("count", ascending=False).show(20, truncate=False)

Bruikbare rijen: 29,419
Unieke categorieën: 1211
Unieke intents: 1556

--- Categorieën ---
+-----------------------------------------------------+-----+
|category                                             |count|
+-----------------------------------------------------+-----+
|ACCOUNT                                              |5986 |
|ORDER                                                |3988 |
|REFUND                                               |2992 |
|CONTACT                                              |1999 |
|INVOICE                                              |1999 |
|PAYMENT                                              |1998 |
|FEEDBACK                                             |1997 |
|DELIVERY                                             |1994 |
|SHIPPING                                             |1970 |
|SUBSCRIPTION                                         |999  |
|CANCEL                                               |950  |
| city                                   

**Observatie:** Er lijken 1.211 unieke categorieën te zijn, maar dat klopt niet.
De echte categorieën (ACCOUNT, ORDER, REFUND, etc.) staan bovenaan in hoofdletters.
Daaronder zien we vervuilde waarden als "city", "street address", "Mastercard" —
dit zijn opnieuw stukken van kapotte antwoorden. We moeten strenger filteren.

### Bitext — Filteren op echte categorieën

We filteren op basis van een simpele regel: echte categorieën zijn volledig in hoofdletters.
Dit scheidt de 11 echte labels van de vervuilde waarden.

In [0]:
# De echte categorieën zijn in HOOFDLETTERS en een enkel woord
# Filter: category moet volledig uppercase zijn
from pyspark.sql.functions import upper, trim

df_bitext_valid = df_bitext_clean.filter(
    trim(col("category")) == upper(trim(col("category")))
)

n_categories = df_bitext_valid.select("category").distinct().count()
n_intents = df_bitext_valid.select("intent").distinct().count()

print(f"Valide rijen: {df_bitext_valid.count():,}")
print(f"Echte categorieën: {n_categories}")
print(f"Echte intents: {n_intents}")
print(f"\n--- Categorieën ---")
df_bitext_valid.groupBy("category").count().orderBy("count", ascending=False).show(20, truncate=False)
print(f"\n--- Intents ---")
df_bitext_valid.groupBy("intent").count().orderBy("count", ascending=False).show(30, truncate=False)

Valide rijen: 26,874
Echte categorieën: 12
Echte intents: 29

--- Categorieën ---
+------------+-----+
|category    |count|
+------------+-----+
|ACCOUNT     |5986 |
|ORDER       |3988 |
|REFUND      |2992 |
|INVOICE     |1999 |
|CONTACT     |1999 |
|PAYMENT     |1998 |
|FEEDBACK    |1997 |
|DELIVERY    |1994 |
|SHIPPING    |1970 |
|SUBSCRIPTION|999  |
|CANCEL      |950  |
| SMS        |2    |
+------------+-----+


--- Intents ---
+------------------------------------------------------+-----+
|intent                                                |count|
+------------------------------------------------------+-----+
|contact_customer_service                              |1000 |
|complaint                                             |1000 |
|switch_account                                        |1000 |
|edit_account                                          |1000 |
|check_invoice                                         |1000 |
|delivery_period                                       |999 

**Observatie:** Na filtering houden we **26.874 valide rijen** over met **11 echte categorieën** en **27 intents**.
De verdeling is goed gebalanceerd: elke intent heeft ~950–1.000 rijen.
ACCOUNT is de grootste categorie (5.986 rijen), CANCEL de kleinste (950 rijen).
Er zitten nog 2 vervuilde intents onderaan — verwaarloosbaar (2 rijen).

## 2.2 Twitter Dataset

We onderzoeken de structuur van de Twitter dataset: 
verdeling klant vs. support, unieke bedrijven, en of we conversatieparen kunnen reconstrueren.

In [0]:
# Verdeling inbound (klant) vs outbound (support)
print("--- Inbound verdeling ---")
df_twitter_raw.groupBy("inbound").count().show()

# Hoeveel unieke bedrijven (authors die NIET inbound zijn)?
support_accounts = df_twitter_raw.filter(col("inbound") == "False")
n_companies = support_accounts.select("author_id").distinct().count()
print(f"Unieke support-accounts (bedrijven): {n_companies}")

# Top 10 bedrijven
print("\n--- Top 10 bedrijven (meeste tweets) ---")
support_accounts.groupBy("author_id").count().orderBy("count", ascending=False).show(10, truncate=False)

--- Inbound verdeling ---
+-----------+-----+
|    inbound|count|
+-----------+-----+
|17004,17006|    1|
|      17623|    1|
|      33305|    1|
|      42044|    1|
|      46520|    1|
|      56975|    1|
|      62377|    1|
|      65786|    1|
|      80186|    1|
|      81798|    1|
|     156324|    1|
|     205075|    1|
|     209902|    2|
|     215147|    1|
|     227143|    1|
|     228282|    1|
|     249146|    1|
|     309126|    1|
|     319784|    1|
|     353562|    1|
+-----------+-----+
only showing top 20 rows
Unieke support-accounts (bedrijven): 108

--- Top 10 bedrijven (meeste tweets) ---
+---------------+------+
|author_id      |count |
+---------------+------+
|AmazonHelp     |169840|
|AppleSupport   |106860|
|Uber_Support   |56270 |
|SpotifyCares   |43265 |
|Delta          |42253 |
|Tesco          |38573 |
|AmericanAir    |36764 |
|TMobileHelp    |34317 |
|comcastcares   |33031 |
|British_Airways|29361 |
+---------------+------+
only showing top 10 rows


**Observatie:** De `inbound`-kolom is vervuild — naast True/False zien we ook getallen.
Dit zijn rijen waar komma's in de tekst de CSV-parser in de war brachten.
De 108 unieke support-accounts vertegenwoordigen bekende bedrijven,
met AmazonHelp (169.840 tweets) en AppleSupport (106.860) als grootste.

### Twitter — Corrupte rijen kwantificeren

We tellen hoeveel rijen een geldig `inbound`-veld hebben (True of False)
versus hoeveel rijen corrupt zijn door verschoven kolommen.

In [0]:
# Hoeveel rijen hebben een correct inbound-veld (True/False)?
correct_inbound = df_twitter_raw.filter(col("inbound").isin("True", "False")).count()
total = df_twitter_raw.count()
corrupt = total - correct_inbound

print(f"Correcte rijen (True/False): {correct_inbound:,}  ({correct_inbound/total*100:.1f}%)")
print(f"Corrupte rijen:              {corrupt:,}  ({corrupt/total*100:.1f}%)")

# Verdeling van de correcte rijen
print("\n--- Verdeling klant vs support ---")
df_twitter_raw.filter(col("inbound").isin("True", "False")) \
    .groupBy("inbound").count() \
    .orderBy("inbound") \
    .show()

Correcte rijen (True/False): 2,811,774  (94.8%)
Corrupte rijen:              154,695  (5.2%)

--- Verdeling klant vs support ---
+-------+-------+
|inbound|  count|
+-------+-------+
|  False|1273931|
|   True|1537843|
+-------+-------+



**Observatie:** 94.8% van de rijen (2.811.774) is correct — slechts 5.2% is corrupt.
De verdeling klant (1.537.843) vs. support (1.273.931) is logisch:
niet elke klantvraag krijgt een antwoord, en sommige support-tweets zijn follow-ups.

### Twitter — Conversatieparen reconstrueren

We onderzoeken of we klant-tweets kunnen koppelen aan hun support-antwoorden.
Dit is essentieel voor het cloud-model: het heeft vraag-antwoord paren nodig om te leren.

In [0]:
# Hoeveel klantttweets hebben een antwoord van support?
df_clean_twitter = df_twitter_raw.filter(col("inbound").isin("True", "False"))

# Klant-tweets die een response_tweet_id hebben
customer_tweets = df_clean_twitter.filter(col("inbound") == "True")
customer_with_response = customer_tweets.filter(col("response_tweet_id").isNotNull()).count()
customer_total = customer_tweets.count()

print(f"Klant-tweets totaal:        {customer_total:,}")
print(f"Met support-antwoord:       {customer_with_response:,}  ({customer_with_response/customer_total*100:.1f}%)")
print(f"Zonder antwoord:            {customer_total - customer_with_response:,}")

# Preview van een conversatiepaar
print("\n--- Voorbeeld conversatiepaar ---")
sample_customer = customer_tweets.filter(col("response_tweet_id").isNotNull()).first()
sample_response_id = sample_customer["response_tweet_id"]

print(f"KLANT (tweet {sample_customer['tweet_id']}):")
print(f"  {sample_customer['text'][:120]}")
print(f"\nSUPPORT (tweet {sample_response_id}):")
response = df_clean_twitter.filter(col("tweet_id") == sample_response_id).first()
if response:
    print(f"  {response['text'][:120]}")
else:
    print("  (antwoord niet gevonden)")

Klant-tweets totaal:        1,537,843
Met support-antwoord:       1,245,030  (81.0%)
Zonder antwoord:            292,813

--- Voorbeeld conversatiepaar ---
KLANT (tweet 3):
  @sprintcare I have sent several private messages and no one is responding as usual

SUPPORT (tweet 1):
  @115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist


**Observatie:** 81% van de klantttweets (1.245.030) heeft een gekoppeld support-antwoord.
Het voorbeeld bevestigt dat de koppeling werkt: een klantvraag over niet-reageren
wordt beantwoord met een aanbod om te helpen via een beveiligd kanaal.

## 2.3 EDA Samenvatting

| | Bitext | Twitter |
|---|---|---|
| **Totaal rijen** | 71.963 | 2.966.469 |
| **Bruikbare rijen** | 26.874 (37%) | 2.811.774 (95%) |
| **Vervuild door** | Multiline antwoorden in CSV | Komma's verschuiven kolommen |
| **Categorieën** | 11 (gelabeld) | Geen labels |
| **Intents** | 27 (gelabeld) | Geen labels |
| **Conversatieparen** | Vraag + antwoord per rij | 1.245.030 koppelbare paren |
| **Doel** | Edge-model (classificatie) | Cloud-model (antwoord genereren) |

## 2.4 Bronze — Ruwe data opslaan als Delta-tabellen

We slaan de ruwe data op als Delta-tabellen in de Bronze-laag.
Dit is de eerste stap van de medallion-architectuur: de originele data ongewijzigd bewaren.
Delta-format is sneller dan CSV (geen herparsen) en ondersteunt versiebeheer.

In [0]:
# Sla ruwe data op als Delta-tabellen (Bronze laag)
# We gebruiken de VOLLEDIGE ruwe data — opschonen doen we pas in Silver

spark.sql("CREATE SCHEMA IF NOT EXISTS flowsure.bronze")

df_bitext_raw.write.mode("overwrite").saveAsTable("flowsure.bronze.bitext_raw")
print("bitext_raw opgeslagen")

df_twitter_raw.write.mode("overwrite").saveAsTable("flowsure.bronze.twitter_raw")
print("twitter_raw opgeslagen")

# Verificatie
for table in ["bitext_raw", "twitter_raw"]:
    count = spark.table(f"flowsure.bronze.{table}").count()
    print(f"flowsure.bronze.{table}: {count:,} rijen")

bitext_raw opgeslagen
twitter_raw opgeslagen
flowsure.bronze.bitext_raw: 71,963 rijen
flowsure.bronze.twitter_raw: 2,966,469 rijen


**Resultaat:** Beide tabellen staan in `flowsure.bronze`. Vanaf nu lezen we altijd
van de Delta-tabellen — de originele CSV's hoeven niet meer aangeraakt te worden.

# 3. ETL Pipeline — Bronze → Silver

## 3.1 Bitext opschonen

We filteren de kapotte rijen eruit en houden alleen rijen met:
- Een geldige categorie (volledig uppercase, uit de bekende 11)
- Een geldige intent  
- Een niet-lege instruction (klantvraag)

In [0]:
from pyspark.sql.functions import col, trim, upper, lit

# Lees van Bronze
df_bitext = spark.table("flowsure.bronze.bitext_raw")

# Bekende categorieën die we in de EDA hebben gevonden
VALID_CATEGORIES = [
    "ACCOUNT", "ORDER", "REFUND", "CONTACT", "INVOICE",
    "PAYMENT", "FEEDBACK", "DELIVERY", "SHIPPING",
    "SUBSCRIPTION", "CANCEL"
]

# Filter: alleen rijen met geldige categorie, intent en instruction
df_bitext_silver = (
    df_bitext
    .filter(col("instruction").isNotNull())
    .filter(col("intent").isNotNull())
    .filter(trim(col("category")).isin(VALID_CATEGORIES))
    .select(
        trim(col("instruction")).alias("text"),
        trim(col("category")).alias("category"),
        trim(col("intent")).alias("intent"),
        trim(col("response")).alias("response"),
        trim(col("flags")).alias("flags"),
    )
)

total_before = df_bitext.count()
total_after = df_bitext_silver.count()
removed = total_before - total_after

print(f"Bronze:  {total_before:,} rijen")
print(f"Silver:  {total_after:,} rijen")
print(f"Verwijderd: {removed:,} rijen ({removed/total_before*100:.1f}%)")

df_bitext_silver.show(5, truncate=80)

Bronze:  71,963 rijen
Silver:  26,872 rijen
Verwijderd: 45,091 rijen (62.7%)
+------------------------------------------------------------+--------+------------+--------------------------------------------------------------------------------+-----+
|                                                        text|category|      intent|                                                                        response|flags|
+------------------------------------------------------------+--------+------------+--------------------------------------------------------------------------------+-----+
|            question about cancelling order {{Order Number}}|   ORDER|cancel_order|I've understood you have a question regarding canceling order {{Order Number}...|    B|
|  i have a question about cancelling oorder {{Order Number}}|   ORDER|cancel_order|I've been informed that you have a question about canceling order {{Order Num...|  BQZ|
|             i need help cancelling puchase {{Order Number}}| 

**Resultaat:** Van 71.963 ruwe rijen houden we 26.872 schone rijen over (37%).
De 62.7% verwijderde rijen waren geen echte tickets maar CSV-artefacten.
De overgebleven data heeft nette kolommen: `text`, `category`, `intent`, `response` en `flags`.

## 3.2 Twitter opschonen

We filteren de corrupte rijen, converteren timestamps, 
verwijderen @mentions uit de tekst, en koppelen klant-tweets aan support-antwoorden 
om conversatieparen te maken.

In [0]:
from pyspark.sql.functions import (
    col, trim, regexp_replace, to_timestamp, when, length
)

# Lees van Bronze
df_twitter = spark.table("flowsure.bronze.twitter_raw")

# Stap 1: Alleen rijen met geldig inbound-veld
df_twitter_valid = df_twitter.filter(col("inbound").isin("True", "False"))

# Stap 2: Tekst opschonen — @mentions verwijderen, extra spaties weg
df_twitter_clean = (
    df_twitter_valid
    .withColumn("text_clean", regexp_replace(col("text"), r"@\S+", ""))
    .withColumn("text_clean", regexp_replace(col("text_clean"), r"\s+", " "))
    .withColumn("text_clean", trim(col("text_clean")))
)

# Stap 3: Timestamp converteren (Spark 4 compatible)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

df_twitter_clean = df_twitter_clean.withColumn(
    "timestamp",
    to_timestamp(col("created_at"), "EEE MMM dd HH:mm:ss Z yyyy")
)

# Stap 4: Inbound naar boolean
df_twitter_clean = df_twitter_clean.withColumn(
    "is_customer",
    when(col("inbound") == "True", True).otherwise(False)
)

# Selecteer en hernoem kolommen
df_twitter_silver = df_twitter_clean.select(
    col("tweet_id"),
    col("author_id"),
    col("is_customer"),
    col("timestamp"),
    col("text_clean").alias("text"),
    col("response_tweet_id"),
    col("in_response_to_tweet_id"),
)

# Filter lege tekst na opschonen
df_twitter_silver = df_twitter_silver.filter(length(col("text")) > 0)

total_before = df_twitter.count()
total_after = df_twitter_silver.count()

print(f"Bronze:  {total_before:,} rijen")
print(f"Silver:  {total_after:,} rijen")
print(f"Verwijderd: {total_before - total_after:,} rijen")

print("\n--- Voorbeeld opgeschoonde tekst ---")
df_twitter_silver.filter(col("is_customer") == True).show(5, truncate=100)

Bronze:  2,966,469 rijen
Silver:  2,796,845 rijen
Verwijderd: 169,624 rijen

--- Voorbeeld opgeschoonde tekst ---
+--------+---------+-----------+-------------------+----------------------------------------------------------------------------------------------------+-----------------+-----------------------+
|tweet_id|author_id|is_customer|          timestamp|                                                                                                text|response_tweet_id|in_response_to_tweet_id|
+--------+---------+-----------+-------------------+----------------------------------------------------------------------------------------------------+-----------------+-----------------------+
|  427751|   216720|       true|2017-10-10 13:26:57|Realmente es vergonzoso que una empresa de la magnitud de deje a los clientes tan desprotegidos a...|             NULL|                 427748|
|  427770|   216724|       true|2017-10-10 13:18:53|Thank you . How can I turn this off to get offers 

**Resultaat:** Van 2.966.469 ruwe rijen houden we 2.796.845 over.
De @mentions zijn verwijderd, timestamps geconverteerd, en `is_customer` is nu een echte boolean.
De opgeschoonde tekst is leesbaar en klaar voor verdere verwerking.

### Silver-tabellen opslaan

We schrijven de opgeschoonde data weg naar de Silver-laag in de Databricks catalog.

In [0]:
# Silver tabellen opslaan
spark.sql("CREATE SCHEMA IF NOT EXISTS flowsure.silver")

df_bitext_silver.write.mode("overwrite").saveAsTable("flowsure.silver.bitext_clean")
print("bitext_clean opgeslagen")

df_twitter_silver.write.mode("overwrite").saveAsTable("flowsure.silver.twitter_clean")
print("twitter_clean opgeslagen")

# Verificatie
for table in ["bitext_clean", "twitter_clean"]:
    count = spark.table(f"flowsure.silver.{table}").count()
    print(f"flowsure.silver.{table}: {count:,} rijen")

bitext_clean opgeslagen
twitter_clean opgeslagen
flowsure.silver.bitext_clean: 26,872 rijen
flowsure.silver.twitter_clean: 2,796,845 rijen


**Resultaat:** Beide Silver-tabellen zijn opgeslagen en geverifieerd.

# 4. Feature Engineering — Silver → Gold

We maken twee Gold-tabellen, één per model:

- **Gold classificatie** (voor edge-model): klanttekst + labels (intent, categorie, prioriteit)
- **Gold conversatieparen** (voor cloud-model): klantvraag gekoppeld aan support-antwoord

## 4.1 Gold — Classificatie features (Bitext → Edge-model)

We voegen een prioriteit-label toe op basis van de intent. 
Intents die gaan over klachten, annuleringen of betalingsproblemen krijgen een hogere prioriteit.

In [0]:
from pyspark.sql.functions import col, when, length, lower

df_bitext = spark.table("flowsure.silver.bitext_clean")

# Prioriteit afleiden uit intent
HIGH_PRIORITY_INTENTS = ["complaint", "payment_issue", "cancel_order", "check_cancellation_fee"]
MEDIUM_PRIORITY_INTENTS = ["track_refund", "get_refund", "check_refund_policy", "delivery_period", "registration_problems"]

df_gold_classification = (
    df_bitext
    .withColumn(
        "priority",
        when(col("intent").isin(HIGH_PRIORITY_INTENTS), "high")
        .when(col("intent").isin(MEDIUM_PRIORITY_INTENTS), "medium")
        .otherwise("low")
    )
    .withColumn("text_length", length(col("text")))
    .select("text", "category", "intent", "priority", "flags", "text_length", "response")
)

print(f"Gold classificatie: {df_gold_classification.count():,} rijen")

print("\n--- Prioriteit verdeling ---")
df_gold_classification.groupBy("priority").count().orderBy("count", ascending=False).show()

print("\n--- Preview ---")
df_gold_classification.show(5, truncate=70)

Gold classificatie: 26,872 rijen

--- Prioriteit verdeling ---
+--------+-----+
|priority|count|
+--------+-----+
|     low|17935|
|  medium| 4990|
|    high| 3947|
+--------+-----+


--- Preview ---
+------------------------------------------------------------+--------+------------+--------+-----+-----------+----------------------------------------------------------------------+
|                                                        text|category|      intent|priority|flags|text_length|                                                              response|
+------------------------------------------------------------+--------+------------+--------+-----+-----------+----------------------------------------------------------------------+
|            question about cancelling order {{Order Number}}|   ORDER|cancel_order|    high|    B|         48|I've understood you have a question regarding canceling order {{Ord...|
|  i have a question about cancelling oorder {{Order Number}}|   ORD

**Resultaat:** 26.872 rijen met prioriteitslabels. De verdeling is realistisch:
66.7% low (bijv. feedback, contact), 18.6% medium (bijv. refund tracking),
en 14.7% high (bijv. klachten, betalingsproblemen).
De text_length feature kan het model helpen: urgente berichten zijn vaak korter en directer.

## 4.2 Gold — Conversatieparen (Twitter → Cloud-model)

We koppelen klant-tweets aan hun support-antwoorden via de tweet_id's.
Dit levert vraag-antwoord paren op waarmee het cloud-model kan leren antwoorden te genereren.

In [0]:
from pyspark.sql.functions import col, regexp_extract
df_twitter = spark.table("flowsure.silver.twitter_clean")

# Klant-tweets (de vragen)
df_customers = (
    df_twitter
    .filter(col("is_customer") == True)
    .filter(col("response_tweet_id").isNotNull())
    # Sommige tweets hebben meerdere response_ids (bijv. "427778,427779")
    # We pakken alleen het eerste antwoord
    .withColumn("first_response_id", regexp_extract(col("response_tweet_id"), r"^(\d+)", 1))
    .select(
        col("tweet_id").alias("customer_tweet_id"),
        col("text").alias("customer_text"),
        col("timestamp").alias("customer_timestamp"),
        col("first_response_id"),
    )
)

# Support-tweets (de antwoorden)
df_support = (
    df_twitter
    .filter(col("is_customer") == False)
    .select(
        col("tweet_id").alias("support_tweet_id"),
        col("text").alias("support_text"),
        col("author_id").alias("company"),
    )
)

# Koppel vraag aan antwoord
df_gold_conversations = (
    df_customers
    .join(df_support, df_customers["first_response_id"] == df_support["support_tweet_id"], "inner")
    .select("customer_tweet_id", "customer_text", "support_text", "company", "customer_timestamp")
)

print(f"Gold conversatieparen: {df_gold_conversations.count():,}")

print("\n--- Voorbeeld paren ---")
df_gold_conversations.select("customer_text", "support_text", "company").show(5, truncate=80)

Gold conversatieparen: 1,078,425

--- Voorbeeld paren ---
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+---------------+
|                                                                   customer_text|                                                                    support_text|        company|
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+---------------+
|. I heard you guys were bad; after an install no-show &amp; 25+ min of being ...|I apologize for the poor experience you've had with us. If there is anything ...|CenturyLinkHelp|
|Thanks to the gate agent at D7 at for being so kind. This airport is great! G...|Hi Jamie! Thanks for the awesome compliment for our PDX staff member. Happy t...|          Delta|
|I'd have to fly your airline again to do 

**Resultaat:** 1.078.425 conversatieparen succesvol gekoppeld.
Elk paar bevat de klantvraag, het support-antwoord, en het bedrijf.
Dit is de trainingsdata voor het cloud-model dat antwoorden leert genereren.

### Gold-tabellen opslaan en catalog verifiëren

We slaan beide Gold-tabellen op en tonen een overzicht van de hele medallion-architectuur.

In [0]:
# Gold tabellen opslaan
spark.sql("CREATE SCHEMA IF NOT EXISTS flowsure.gold")

df_gold_classification.write.mode("overwrite").saveAsTable("flowsure.gold.classification_features")
print("classification_features opgeslagen")

df_gold_conversations.write.mode("overwrite").saveAsTable("flowsure.gold.conversation_pairs")
print("conversation_pairs opgeslagen")

# Verificatie van de hele medallion-architectuur
print("\n--- Flowsure Catalog Overzicht ---")
for schema in ["bronze", "silver", "gold"]:
    tables = spark.sql(f"SHOW TABLES IN flowsure.{schema}").collect()
    for t in tables:
        count = spark.table(f"flowsure.{schema}.{t.tableName}").count()
        print(f"  flowsure.{schema}.{t.tableName}: {count:,} rijen")

classification_features opgeslagen
conversation_pairs opgeslagen

--- Flowsure Catalog Overzicht ---
  flowsure.bronze.bitext_raw: 71,963 rijen
  flowsure.bronze.twitter_raw: 2,966,469 rijen
  flowsure.silver.bitext_clean: 26,872 rijen
  flowsure.silver.twitter_clean: 2,796,845 rijen
  flowsure.gold.classification_features: 26,872 rijen
  flowsure.gold.conversation_pairs: 1,078,425 rijen


**Resultaat:** De volledige medallion-architectuur staat: Bronze (ruwe data),
Silver (opgeschoond), Gold (modelklaar). Elke laag is traceerbaar en reproduceerbaar.

# 5. Streaming Pipeline — Simulatie

We simuleren live ticket-instroom met Spark Structured Streaming.
In plaats van Kafka gebruiken we file-based streaming: 
we droppen ruwe data als JSON-bestanden in een map, 
en de streaming job pikt ze automatisch op en verwerkt ze 
met dezelfde opschoonlogica als de batch-pipeline.

## 5.1 Simulatiedata klaarzetten

We nemen een stukje ruwe Twitter-data en knippen het op in kleine JSON-bestanden.
Elk bestand simuleert een "batch" nieuwe tickets die binnenkomen.

In [0]:
# Pak 1000 ruwe rijen met geldig inbound-veld als simulatiedata
df_sim = (
    spark.table("flowsure.bronze.twitter_raw")
    .filter(col("inbound").isin("True", "False"))
    .limit(1000)
)

# Maak de streaming input map leeg en schrijf als JSON-bestanden
# repartition(5) = 5 losse bestanden, alsof er 5 "batches" binnenkomen
streaming_input_path = "/Volumes/flowsure/default/raw_files/streaming_input"

dbutils.fs.rm(streaming_input_path, recurse=True)
dbutils.fs.mkdirs(streaming_input_path)

df_sim.repartition(5).write.mode("overwrite").json(streaming_input_path)

# Check wat er in de map staat
files = dbutils.fs.ls(streaming_input_path)
for f in files:
    print(f"{f.name}  ({f.size:,} bytes)")

print(f"\nTotaal: {len(files)} bestanden met {df_sim.count():,} rijen")

_SUCCESS  (0 bytes)
_committed_8421626052149174031  (474 bytes)
_started_8421626052149174031  (0 bytes)
part-00000-tid-8421626052149174031-8276fce8-d8bd-4158-a4e4-e62805243126-225-1-c000.json  (52,532 bytes)
part-00001-tid-8421626052149174031-8276fce8-d8bd-4158-a4e4-e62805243126-226-1-c000.json  (52,435 bytes)
part-00002-tid-8421626052149174031-8276fce8-d8bd-4158-a4e4-e62805243126-229-1-c000.json  (53,163 bytes)
part-00003-tid-8421626052149174031-8276fce8-d8bd-4158-a4e4-e62805243126-227-1-c000.json  (53,439 bytes)
part-00004-tid-8421626052149174031-8276fce8-d8bd-4158-a4e4-e62805243126-228-1-c000.json  (53,422 bytes)

Totaal: 8 bestanden met 1,000 rijen


**Resultaat:** 5 JSON-bestanden aangemaakt, elk ~52 KB. Samen bevatten ze 1.000 ruwe tweets
die fungeren als gesimuleerde nieuwe tickets.

## 5.2 Streaming schema definiëren

Structured Streaming moet vooraf weten welke kolommen en types de data heeft.
Bij batch kan Spark dit zelf raden (inferSchema), maar bij streaming moet je het expliciet opgeven.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

# Zelfde structuur als de ruwe Twitter data
twitter_schema = StructType([
    StructField("tweet_id", StringType(), True),
    StructField("author_id", StringType(), True),
    StructField("inbound", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("text", StringType(), True),
    StructField("response_tweet_id", StringType(), True),
    StructField("in_response_to_tweet_id", StringType(), True),
])

print("Schema gedefinieerd:")
print(twitter_schema.simpleString())

Schema gedefinieerd:
struct<tweet_id:string,author_id:string,inbound:string,created_at:string,text:string,response_tweet_id:string,in_response_to_tweet_id:string>


## 5.3 Streaming job — lezen, opschonen, wegschrijven

We starten een Structured Streaming query die:
1. De JSON-map bewaakt voor nieuwe bestanden
2. Dezelfde opschoonlogica toepast als de batch-pipeline (stap 3.2)
3. Het resultaat continu wegschrijft naar een Delta-tabel

In [0]:
from pyspark.sql.functions import col, trim, regexp_replace, when, length, current_timestamp, udf
from pyspark.sql.types import TimestampType
from datetime import datetime

streaming_input_path = "/Volumes/flowsure/default/raw_files/streaming_input"
checkpoint_path = "/Volumes/flowsure/default/raw_files/streaming_checkpoint"

# Ruim oude checkpoint op zodat we vers beginnen
dbutils.fs.rm(checkpoint_path, recurse=True)

# UDF om het Twitter timestamp-formaat te parsen
@udf(TimestampType())
def parse_twitter_ts(ts_string):
    if ts_string is None:
        return None
    try:
        return datetime.strptime(ts_string, "%a %b %d %H:%M:%S %z %Y")
    except Exception:
        return None

# Stap 1: Lees de map als stream
df_stream = (
    spark.readStream
    .schema(twitter_schema)
    .json(streaming_input_path)
)

# Stap 2: Zelfde opschoonlogica als batch, maar met UDF voor timestamp
df_stream_clean = (
    df_stream
    .filter(col("inbound").isin("True", "False"))
    .withColumn("text_clean", regexp_replace(col("text"), r"@\S+", ""))
    .withColumn("text_clean", regexp_replace(col("text_clean"), r"\s+", " "))
    .withColumn("text_clean", trim(col("text_clean")))
    .withColumn("timestamp", parse_twitter_ts(col("created_at")))
    .withColumn("is_customer", when(col("inbound") == "True", True).otherwise(False))
    .withColumn("processed_at", current_timestamp())
    .filter(length(col("text_clean")) > 0)
    .select(
        col("tweet_id"),
        col("author_id"),
        col("is_customer"),
        col("timestamp"),
        col("text_clean").alias("text"),
        col("processed_at"),
    )
)

# Stap 3: Schrijf naar Delta-tabel
query = (
    df_stream_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("flowsure.gold.streaming_processed")
)

query.awaitTermination()
print("Streaming batch verwerkt!")

Streaming batch verwerkt!


**Resultaat:** De streaming job heeft alle 5 JSON-bestanden verwerkt via `readStream`.
Dezelfde opschoonlogica (mentions verwijderen, timestamps parsen, lege tekst filteren)
is toegepast — maar dan in streaming-modus. De checkpoint onthoudt welke bestanden al verwerkt zijn.

### Streaming resultaat verifiëren

We controleren hoeveel rijen de streaming job heeft verwerkt en bekijken een preview.

In [0]:
# Check de resultaten
df_streamed = spark.table("flowsure.gold.streaming_processed")

print(f"Streaming verwerkt: {df_streamed.count():,} rijen")
print(f"\n--- Preview ---")
df_streamed.show(5, truncate=80)

Streaming verwerkt: 994 rijen

--- Preview ---
+--------+----------+-----------+-------------------+--------------------------------------------------------------------------------+-----------------------+
|tweet_id| author_id|is_customer|          timestamp|                                                                            text|           processed_at|
+--------+----------+-----------+-------------------+--------------------------------------------------------------------------------+-----------------------+
|  400905|    216964|       true|2017-10-10 07:56:35|                                    Can you get the Siri watch face on iwatch2 ?|2026-03-18 15:48:52.241|
|  427371|    216646|       true|2017-10-10 13:02:40|You didn't call on the number which I gave on my shipping Address . I want re...|2026-03-18 15:48:52.241|
|  427771|AmazonHelp|      false|2017-10-10 13:23:00|We can't access your order info from here but couriers deliver until 21:00. L...|2026-03-18 15:48:52.241|

**Resultaat:** 994 van de 1.000 rijen zijn verwerkt (6 weggevallen door lege tekst na opschonen).
De `processed_at` kolom toont het tijdstip van verwerking —
handig om te monitoren wanneer tickets zijn binnengekomen vs. verwerkt.

## 5.4 Streaming samenvatting

De streaming pipeline verwerkt ruwe tickets met **dezelfde opschoonlogica** als de batch-pipeline.
Het enige verschil in code is `readStream` i.p.v. `read` en `writeStream` i.p.v. `write`.

In productie zou dit als volgt werken:
- De `streaming_input` map wordt vervangen door Kafka of een ander berichtsysteem
- De job wordt periodiek gescheduled (bijv. elk uur via Databricks Workflows)
- De checkpoint zorgt ervoor dat alleen **nieuwe** bestanden worden verwerkt
- De architectuur blijft identiek — alleen de bron verandert

# 6. Data Monitoring — Drift Baseline

We leggen de verdeling van de trainingsdata vast als referentiepunt.
Als toekomstige data significant afwijkt van deze baseline 
(bijv. nieuwe categorieën, andere taalverdeling, kortere/langere teksten), 
is dat een signaal om het model opnieuw te trainen.

In [0]:
# Baseline statistieken van de Gold classificatie-data (edge-model)
df_gold = spark.table("flowsure.gold.classification_features")

print("=== DRIFT BASELINE — Classificatie Data ===\n")

# 1. Verdeling per categorie (percentages)
total = df_gold.count()
print(f"Totaal rijen: {total:,}\n")

print("--- Categorie verdeling ---")
df_gold.groupBy("category").count() \
    .withColumn("percentage", (col("count") / total * 100).cast("decimal(5,1)")) \
    .orderBy("count", ascending=False) \
    .show(15, truncate=False)

# 2. Verdeling per prioriteit
print("--- Prioriteit verdeling ---")
df_gold.groupBy("priority").count() \
    .withColumn("percentage", (col("count") / total * 100).cast("decimal(5,1)")) \
    .orderBy("count", ascending=False) \
    .show()

# 3. Tekstlengte statistieken
print("--- Tekstlengte statistieken ---")
df_gold.select("text_length").summary("mean", "stddev", "min", "25%", "50%", "75%", "max").show()

=== DRIFT BASELINE — Classificatie Data ===

Totaal rijen: 26,872

--- Categorie verdeling ---
+------------+-----+----------+
|category    |count|percentage|
+------------+-----+----------+
|ACCOUNT     |5986 |22.3      |
|ORDER       |3988 |14.8      |
|REFUND      |2992 |11.1      |
|INVOICE     |1999 |7.4       |
|CONTACT     |1999 |7.4       |
|PAYMENT     |1998 |7.4       |
|FEEDBACK    |1997 |7.4       |
|DELIVERY    |1994 |7.4       |
|SHIPPING    |1970 |7.3       |
|SUBSCRIPTION|999  |3.7       |
|CANCEL      |950  |3.5       |
+------------+-----+----------+

--- Prioriteit verdeling ---
+--------+-----+----------+
|priority|count|percentage|
+--------+-----+----------+
|     low|17935|      66.7|
|  medium| 4990|      18.6|
|    high| 3947|      14.7|
+--------+-----+----------+

--- Tekstlengte statistieken ---
+-------+------------------+
|summary|       text_length|
+-------+------------------+
|   mean| 46.88951324799047|
| stddev|10.897578128500571|
|    min|           

**Resultaat:** De baseline is vastgelegd. ACCOUNT is de dominante categorie (22.3%),
gevolgd door ORDER (14.8%). Twee derde van de tickets is low-priority.
De gemiddelde tekstlengte is ~47 tekens met een standaardafwijking van ~11.
Als in de toekomst de verdeling significant verschuift (bijv. plotseling 40% PAYMENT),
is dat een drift-signaal en moet het model hertraind worden.

### Baselines opslaan

We slaan de verdelingen op als Delta-tabellen zodat we ze later kunnen vergelijken met nieuwe data.

In [0]:
# Sla de baseline op als Delta-tabel
from pyspark.sql.functions import lit, current_timestamp

# Categorie baseline
df_cat_baseline = (
    df_gold.groupBy("category").count()
    .withColumn("percentage", (col("count") / total * 100))
    .withColumn("baseline_date", current_timestamp())
)

# Prioriteit baseline
df_prio_baseline = (
    df_gold.groupBy("priority").count()
    .withColumn("percentage", (col("count") / total * 100))
    .withColumn("baseline_date", current_timestamp())
)

spark.sql("CREATE SCHEMA IF NOT EXISTS flowsure.monitoring")

df_cat_baseline.write.mode("overwrite").saveAsTable("flowsure.monitoring.baseline_categories")
df_prio_baseline.write.mode("overwrite").saveAsTable("flowsure.monitoring.baseline_priorities")

print("Baselines opgeslagen:")
print("  flowsure.monitoring.baseline_categories")
print("  flowsure.monitoring.baseline_priorities")

Baselines opgeslagen:
  flowsure.monitoring.baseline_categories
  flowsure.monitoring.baseline_priorities


# 7. Eindoverzicht

We tonen een compleet overzicht van alle tabellen in de Flowsure catalog
om te verifiëren dat de hele pipeline correct heeft gedraaid.

In [0]:
# Eindoverzicht — alles wat we gebouwd hebben
print("=" * 60)
print("FLOWSURE DATA PIPELINE — COMPLEET OVERZICHT")
print("=" * 60)

for schema in ["bronze", "silver", "gold", "monitoring"]:
    print(f"\n--- flowsure.{schema} ---")
    tables = spark.sql(f"SHOW TABLES IN flowsure.{schema}").collect()
    for t in tables:
        count = spark.table(f"flowsure.{schema}.{t.tableName}").count()
        print(f"  {t.tableName}: {count:,} rijen")

FLOWSURE DATA PIPELINE — COMPLEET OVERZICHT

--- flowsure.bronze ---
  bitext_raw: 71,963 rijen
  twitter_raw: 2,966,469 rijen

--- flowsure.silver ---
  bitext_clean: 26,872 rijen
  twitter_clean: 2,796,845 rijen

--- flowsure.gold ---
  classification_features: 26,872 rijen
  conversation_pairs: 1,078,425 rijen
  streaming_processed: 994 rijen

--- flowsure.monitoring ---
  baseline_categories: 11 rijen
  baseline_priorities: 3 rijen


# 7. Conclusie & Volgende Stappen

## Wat we hebben gebouwd

Een end-to-end datapipeline die twee ruwe datasets transformeert naar modelklare output, 
met een medallion-architectuur (Bronze → Silver → Gold) op Databricks.

| Laag | Tabel | Rijen | Doel |
|---|---|---|---|
| Bronze | bitext_raw | 71.963 | Ruwe CSV, ongewijzigd |
| Bronze | twitter_raw | 2.966.469 | Ruwe CSV, ongewijzigd |
| Silver | bitext_clean | 26.872 | Gevalideerd, kapotte rijen verwijderd |
| Silver | twitter_clean | 2.796.845 | Corrupte rijen gefilterd, tekst opgeschoond, timestamps geconverteerd |
| Gold | classification_features | 26.872 | Intent + categorie + prioriteit → edge-model |
| Gold | conversation_pairs | 1.078.425 | Vraag-antwoord paren → cloud-model |
| Gold | streaming_processed | 994 | Live tickets verwerkt via Structured Streaming |
| Monitoring | baseline_categories | 11 | Referentieverdeling voor drift detectie |
| Monitoring | baseline_priorities | 3 | Referentieverdeling voor drift detectie |

## Leerdoelen gedekt

- **Leerdoel 1** — End-to-end datapipeline: CSV → validatie → transformatie → modelklare output
- **Leerdoel 2** — Batch (3M rijen met Spark) én streaming (Structured Streaming met trigger) 
- **Leerdoel 5** — Drift baseline vastgelegd voor continue monitoring

